# GTZAN Music Genre Classification - Sprint 1: Data Loading & Feature Extraction

## Objective
Build a metadata-only PostgreSQL database for the GTZAN Music Genre clasification.

## Goal

By the end of this notebook:
- ✓ 2 tables created: `music_genre` and `audio_track`
- ✓ 1 view created: `vw_clean_tracks` (filters corrupted and duplicate tracks)
- ✓ 1,000 tracks scanned and loaded with metadata only 
- ✓ Corrupted and duplicate tracks flagged and excluded
- ✓ Stratified 70/15/15 train/val/test split assigned and stored in database
- ✓ Database ready for Sprint 2 PyTorch Dataset class

## Workflow 
1. → Imports
2. → Load .env credentials
3. → Test connection
4. → Dataset path
5. → Create music_genre table
6. → Create audio_track table
7. → Create vw_clean_tracks view
8. → Insert genres
9. → Scan & insert 1000 tracks
10. → Flag corrupted & duplicates
11. → Assign train/val/test split
12. → Final verification

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# PostgreSQL connectivity
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Audio processing
import librosa
# Hashing for  data integrity
import hashlib
#importing random seeds=42
import random

import warnings
warnings.filterwarnings('ignore')

# Suppress audioread macOS warnings
import logging
logging.getLogger('audioread').setLevel(logging.ERROR)

print('✓ Libraries loaded')

✓ Libraries loaded


In [2]:
# ── Load credentials from .env ─────────────────────────────────────────────
load_dotenv()

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = 'localhost'
DB_PORT     = 5432
DB_NAME     = 'music_genre_db'

assert DB_USER,     'DB_USER not found in .env'
assert DB_PASSWORD, 'DB_PASSWORD not found in .env'

print(f'✓ Credentials loaded: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

✓ Credentials loaded: bfanta04@localhost:5432/music_genre_db


In [3]:
# ── Create SQLAlchemy engine and test connection ────────────────────────
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Test the connection
try:
    with engine.connect() as conn:
        result = conn.execute(text('SELECT current_database(), current_user;'))
        db_info = result.fetchone()
        print(f'✓ Connected to: {db_info[0]} as {db_info[1]}')
except Exception as e:
    print(f'✗ Connection failed: {e}')

✓ Connected to: music_genre_db as bfanta04


In [4]:
# Define the path to the GTZAN dataset 
GTZAN_PATH = Path('../Data_Music/processed')

# Check if the path exist
if GTZAN_PATH.exists():
    print(f'✓ Dataset found at: {GTZAN_PATH.absolute()}')
else:
    print(f'✗ Dataset NOT found at: {GTZAN_PATH.absolute()}')
    print(f'Check that the folder exists in your project')

# List the genres
genres = sorted([d.name for d in GTZAN_PATH.iterdir() if d.is_dir()])
print(f'\nGenres found: {len(genres)}')
for genre in genres:
    print(f'  - {genre}')

✓ Dataset found at: /Users/sa19/class-projects/music-genre-classification/Sql_Workflow/../Data_Music/processed

Genres found: 10
  - blues
  - classical
  - country
  - disco
  - hiphop
  - jazz
  - metal
  - pop
  - reggae
  - rock


## Make sure you delete old tables and your database are clean.

In [5]:
# Verify tables in database 
result = pd.read_sql("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_name
""", engine)

print(f'Tables in music_genre_db: {len(result)}')
print(result)

Tables in music_genre_db: 3
        table_name
0      audio_track
1      music_genre
2  vw_clean_tracks


In [6]:
# Verify tables in database 
DROP_EXISTING_TABLES = """
DROP VIEW IF EXISTS vw_clean_tracks;
DROP TABLE IF EXISTS audio_track;
DROP TABLE IF EXISTS music_genre;
"""

with engine.begin() as conn:
    conn.execute(text(DROP_EXISTING_TABLES))
    print('✓ View vw_clean_tracks and tables audio_track and music_genre dropped')

✓ View vw_clean_tracks and tables audio_track and music_genre dropped


# Create Tables 

Here we are going to create the two tables for our dataset **`music_genre`** and **`audio_track`**


In [7]:
# Create music_genre table
CREATE_GENRE_TABLE = """
CREATE TABLE music_genre (
    genre_id   SERIAL PRIMARY KEY,
    genre_name VARCHAR(20) NOT NULL UNIQUE
);
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_GENRE_TABLE))
    print('✓ Table music_genre created')

✓ Table music_genre created


## Explaining rows in table **`audiotrack`**

- file_hash → detects duplicates between files
-  duplicate_flagged → marks duplicate tracks
-  corrupted_flagged → marks corrupted tracks (like jazz.00054.wav)
- split → saves train/val/test to the database
- duration_sec → verifies that all tracks are 30 seconds long
- sample_rate → verifies that all tracks are 22050 Hz
- file_size_bytes → physical file size information


In [8]:
# Create audio_track table 
CREATE_TRACK_TABLE = """
CREATE TABLE audio_track (
    track_id         SERIAL PRIMARY KEY,
    genre_id         INT NOT NULL REFERENCES music_genre(genre_id) ON DELETE CASCADE,
    file_path        TEXT NOT NULL,
    file_name        VARCHAR(64) NOT NULL,
    duration_sec     NUMERIC,
    sample_rate      INT,
    channels         INT,
    file_size_bytes  BIGINT,
    file_hash        CHAR(32),
    duplicate_flagged BOOLEAN DEFAULT FALSE,
    corrupted_flagged BOOLEAN DEFAULT FALSE,
    split            VARCHAR(5),
    created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_TRACK_TABLE))
    print('✓ Table audio_track created')

✓ Table audio_track created


## View: vw_clean_tracks

- This view filters out corrupted and duplicate tracks from audio_track.
- It joins music_genre to include the genre label (e.g. 'blues', 'jazz').
- Used by PyTorch Dataset class to load only clean, valid tracks.
- Query example: SELECT * FROM vw_clean_tracks WHERE split = 'train'

In [9]:
# Create vw_clean_tracks view 
CREATE_VIEW = """
CREATE OR REPLACE VIEW vw_clean_tracks AS
    SELECT 
        at.track_id,
        at.file_path,
        mg.genre_name AS label,
        at.genre_id,
        at.split,
        at.duration_sec,
        at.sample_rate,
        at.channels
    FROM audio_track at
    JOIN music_genre mg ON at.genre_id = mg.genre_id
    WHERE at.corrupted_flagged = FALSE
    AND   at.duplicate_flagged = FALSE;
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_VIEW))
    print('✓ View vw_clean_tracks created')

✓ View vw_clean_tracks created


In [10]:
#let's check the tables are created 

# Verify tables and view created 
result = pd.read_sql("""
    SELECT table_name, table_type
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_name
""", engine)

print(result)

        table_name  table_type
0      audio_track  BASE TABLE
1      music_genre  BASE TABLE
2  vw_clean_tracks        VIEW


# Insert 10 genres in a table **`music_genre`**

In [11]:
# Insert the 10 genres into music_genre 
INSERT_GENRES = """
INSERT INTO music_genre (genre_name) VALUES
('blues'),
('classical'),
('country'),
('disco'),
('hiphop'),
('jazz'),
('metal'),
('pop'),
('reggae'),
('rock')
ON CONFLICT (genre_name) DO NOTHING;
"""

with engine.begin() as conn:
    conn.execute(text(INSERT_GENRES))

# Verify
df_genres = pd.read_sql('SELECT * FROM music_genre ORDER BY genre_name', engine)
print(f'✓ {len(df_genres)} genres inserted')
print(df_genres)

✓ 10 genres inserted
   genre_id genre_name
0         1      blues
1         2  classical
2         3    country
3         4      disco
4         5     hiphop
5         6       jazz
6         7      metal
7         8        pop
8         9     reggae
9        10       rock


# Scan genres_original folder and insert tracks 
Scans each genre folder inside genres_original/ and collects metadata for every .wav file found. For each file we record:
- file_path and file_name  → so PyTorch knows where to load it
- file_size_bytes          → physical file information
- file_hash (MD5)          → to detect duplicate files
- corrupted_flagged        → will be updated after validation
- duplicate_flagged        → will be updated after duplicate check
- split                    → train/val/test (assigned in next step)

In [12]:
# Scan genres_original folder and insert tracks 

def get_file_hash(file_path):
    """Generate MD5 hash to detect duplicates"""
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

# Get genre_id mapping from DB
genre_map = pd.read_sql(
    'SELECT genre_id, genre_name FROM music_genre', engine
).set_index('genre_name')['genre_id'].to_dict()

tracks = []
print('Scanning dataset...\n')

for genre_name, genre_id in genre_map.items():
    genre_path = GTZAN_PATH / genre_name
    wav_files  = sorted(genre_path.glob('*.wav'))
    
    for wav_file in wav_files:
        try:
            stat = wav_file.stat()
            tracks.append({
                'genre_id':         genre_id,
                'file_path':        str(wav_file),
                'file_name':        wav_file.name,
                'file_size_bytes':  stat.st_size,
                'file_hash':        get_file_hash(wav_file),
                'corrupted_flagged': False,
                'duplicate_flagged': False,
                'split':            None,
                'duration_sec':     None,
                'sample_rate':      None,
                'channels':         None,
            })
        except Exception as e:
            print(f'  ✗ Error: {wav_file.name}: {e}')

print(f'✓ {len(tracks)} tracks found')

Scanning dataset...

✓ 9981 tracks found


In [13]:
# Insert tracks into audio_track table 
df_tracks = pd.DataFrame(tracks)

df_tracks.to_sql('audio_track', engine, if_exists='append', index=False)

print(f'✓ {len(df_tracks)} tracks inserted into audio_track')

✓ 9981 tracks inserted into audio_track


In [14]:
# Flag corrupted and duplicate tracks 

# Flag corrupted: jazz.00054.wav (known corrupted file in music dataset)
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE audio_track 
        SET corrupted_flagged = TRUE
        WHERE file_name = 'jazz.00054.wav'
    """))
    print('✓ Corrupted tracks flagged')

# Flag duplicates: same file_hash = duplicate file
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE audio_track
        SET duplicate_flagged = TRUE
        WHERE file_hash IN (
            SELECT file_hash
            FROM audio_track
            GROUP BY file_hash
            HAVING COUNT(*) > 1
        )
    """))
    print('✓ Duplicate tracks flagged')

✓ Corrupted tracks flagged
✓ Duplicate tracks flagged


## Train/Validation/Test Split

we decide to split the data in 3 sets using a 70/15/15 ratio with seed 42 
to ensure all teammates get identical splits.

- train (70%) → model learns 
- validation (15%) → checks if model is improving during training 
- test (15%) → final evaluation after training is done

We believe it is balance keep in mind we have 999 tracks and it will be divide 

- 70% → 699 tracks for training
- 15% → 150 tracks for validation 
- 15% → 150 tracks for testing


In [15]:
#random seeds to make sure everyone in the team has same split 

# Assigning train/val/test split (70/15/15) 
random.seed(42)

# Get all clean tracks grouped by genre
df_clean = pd.read_sql("""
    SELECT track_id, genre_id 
    FROM audio_track
    WHERE corrupted_flagged = FALSE
    AND   duplicate_flagged = FALSE
    ORDER BY genre_id, track_id
""", engine)

print(f'Clean tracks: {len(df_clean)}')

# Assign split per genre (stratified)
splits = []
for genre_id, group in df_clean.groupby('genre_id'):
    track_ids = group['track_id'].tolist()
    random.shuffle(track_ids)
    
    n = len(track_ids)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)
    
    for i, track_id in enumerate(track_ids):
        if i < n_train:
            split = 'train'
        elif i < n_train + n_val:
            split = 'val'
        else:
            split = 'test'
        splits.append({'track_id': track_id, 'split': split})

# Update database
with engine.begin() as conn:
    for row in splits:
        conn.execute(text("""
            UPDATE audio_track 
            SET split = :split 
            WHERE track_id = :track_id
        """), row)

print('✓ Split assigned')

# Verify
df_split = pd.read_sql("""
    SELECT split, COUNT(*) as count 
    FROM audio_track 
    WHERE corrupted_flagged = FALSE
    GROUP BY split
    ORDER BY split
""", engine)
print(df_split)



Clean tracks: 9695
✓ Split assigned
   split  count
0   test   1462
1  train   6783
2    val   1450
3   None    286


In [16]:
# ── Set split = NULL for corrupted and duplicate tracks ─────────────────
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE audio_track
        SET split = NULL
        WHERE corrupted_flagged = TRUE
        OR    duplicate_flagged = TRUE
    """))
    print('✓ Excluded tracks set to NULL')

✓ Excluded tracks set to NULL


In [17]:
# Verify split distribution 
df_split = pd.read_sql("""
    SELECT 
        COALESCE(split, 'excluded') as split, 
        COUNT(*) as count 
    FROM audio_track 
    GROUP BY split
    ORDER BY split
""", engine)

print(f'Total tracks: 1000\n')
print(df_split)

Total tracks: 1000

      split  count
0  excluded    286
1      test   1462
2     train   6783
3       val   1450


## Summary Train/Val/Test

After scanning all 1,000 tracks we identified two data quality issues:

1. Corrupted file **`jazz.00054.wav`** fails to load with Librosa.
2. 28 tracks share the same MD5 hash identical audio content that 
   could confuse the model during training.
3. We set `split = NULL` to exclude corrupted and duplicate tracks 
   from train/val/test.
4. Final usable tracks: **971** divided into:
   - **Train** = 677
   - **Val** = 141
   - **Test** = 153

In [18]:
# Final verification 
print('=== DATABASE STATUS ===\n')

# Count genres
genres = pd.read_sql('SELECT COUNT(*) as count FROM music_genre', engine)
print(f'Genres:          {genres["count"].values[0]}')

# Count all tracks
tracks = pd.read_sql('SELECT COUNT(*) as count FROM audio_track', engine)
print(f'Total tracks:    {tracks["count"].values[0]}')

# Count clean tracks
clean = pd.read_sql('SELECT COUNT(*) as count FROM vw_clean_tracks', engine)
print(f'Clean tracks:    {clean["count"].values[0]}')

# Split distribution
print('\n=== SPLIT DISTRIBUTION ===\n')
df_split = pd.read_sql("""
    SELECT 
        COALESCE(split, 'excluded') as split,
        COUNT(*) as count
    FROM audio_track
    GROUP BY split
    ORDER BY split
""", engine)
print(df_split)

# Sample from view
print('\n=== SAMPLE FROM vw_clean_tracks ===\n')
sample = pd.read_sql("""
    SELECT track_id, file_path, label, split
    FROM vw_clean_tracks
    LIMIT 5
""", engine)
print(sample)

=== DATABASE STATUS ===

Genres:          10
Total tracks:    9981
Clean tracks:    9695

=== SPLIT DISTRIBUTION ===

      split  count
0  excluded    286
1      test   1462
2     train   6783
3       val   1450

=== SAMPLE FROM vw_clean_tracks ===

   track_id                                         file_path  label  split
0      3963  ../Data_Music/processed/disco/disco.00096.08.wav  disco  train
1      3967  ../Data_Music/processed/disco/disco.00097.02.wav  disco  train
2      3972  ../Data_Music/processed/disco/disco.00097.07.wav  disco  train
3      3974  ../Data_Music/processed/disco/disco.00097.09.wav  disco  train
4      3954  ../Data_Music/processed/disco/disco.00095.09.wav  disco  train


In [19]:
# Update duration_sec, sample_rate, channels 
import wave

df_tracks_db = pd.read_sql(
    'SELECT track_id, file_path FROM audio_track', engine
)

print('Reading audio metadata...')

for _, row in df_tracks_db.iterrows():
    try:
        with wave.open(row['file_path'], 'r') as w:
            sample_rate  = w.getframerate()
            channels     = w.getnchannels()
            n_frames     = w.getnframes()
            duration_sec = round(n_frames / sample_rate, 2)

        with engine.begin() as conn:
            conn.execute(text("""
                UPDATE audio_track
                SET duration_sec = :duration_sec,
                    sample_rate  = :sample_rate,
                    channels     = :channels
                WHERE track_id   = :track_id
            """), {
                'duration_sec': duration_sec,
                'sample_rate':  sample_rate,
                'channels':     channels,
                'track_id':     row['track_id']
            })
    except Exception as e:
        print(f'  ✗ {row["file_path"]}: {e}')

print('✓ Audio metadata updated')

Reading audio metadata...
✓ Audio metadata updated


In [20]:
# ── Verify audio metadata ────────────────────────────────────────────────
df_verify = pd.read_sql("""
    SELECT 
        ROUND(AVG(duration_sec), 2) as avg_duration,
        MIN(duration_sec)           as min_duration,
        MAX(duration_sec)           as max_duration,
        AVG(sample_rate)            as sample_rate,
        AVG(channels)               as channels,
        COUNT(*)                    as total_tracks,
        COUNT(duration_sec)         as tracks_with_duration
    FROM audio_track
""", engine)
print(df_verify)

   avg_duration  min_duration  max_duration  sample_rate  channels  \
0           3.0           3.0           3.0      22050.0       1.0   

   total_tracks  tracks_with_duration  
0          9981                  9981  
